<a href="https://colab.research.google.com/github/MugdhoAI/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MugdhoAI/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why


My lane is Lane 2: Refresh / Content Opportunity Scoring. The question is: which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring? I chose this lane because it is the workflow I already ran in the starter notebooks: a hand rule for stale and visible pages, ranked against a small decision tree and scored with Precision@K. I have seen the ranked queue idea work end to end on the starter data, and I would rather spend the next weeks making that ranking honest and useful than start from zero. The output is also concrete: a ranked list of pages with a reason for each one, which a content team can act on directly.


## 2. The question: decision, action, cost of a wrong call

The decision: A content team cannot review every page. Each week, someone must decide which pages get reviewed first for refresh, expansion, protection, pruning, or monitoring. Today that choice is mostly gut feeling or whoever asks loudest. My work improves that one decision.

Who acts: A content editor or SEO specialist works from my ranked queue, starting at the top. For each page, the queue gives a suggested action and the reasons behind it, so the human still makes the final call, but in a much better order.

The output and the unit: One ranked list, one row per page. The unit of analysis is a page. Each page gets a score for how strongly it looks like it needs review, and I will judge the ranking with Precision@K: of the top K pages flagged, how many are actually declining.

Cost of a wrong call: A false alarm wastes scarce editor hours refreshing a healthy page, and an unnecessary rewrite can even hurt a page that was fine. A miss costs the other way: a page that is quietly declining keeps losing traffic while nobody looks at it.

Why data and ML help at all: There are thousands of pages and many signals at once: staleness, impressions, average position, CTR, content age. A hand rule catches the obvious cases at the top, I saw that in the starter notebooks, but it runs out of signal deeper in the list. A simple, readable model can weigh the signals together, as long as it never uses leaked signals such as trend_pct.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter data, computed in the code cell below:

1. Of 30,000 pages, 54.2% are declining, about 16,260 pages. The problem is far too large for manual review, so the order in which pages get reviewed decides most of the value.
2. The obvious hand rule, not updated for 180+ days and still visible with 500+ impressions, finds only 17 pages, but 94.1% of them are declining. The rule is sharp, it is just tiny: it catches the most obvious cases and misses almost everything else.
3. Scored as a ranking, the hand rule reaches Precision@50 of 0.68, against 0.54 for picking pages at random. The signal is real but weak past the very top, because every page outside the small pool ties at a score of zero. A learned, readable ranking that can order the whole inventory is the gap this lane exists to close.


In [7]:
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"1) Pages in the starter data: {len(df):,}")
print(f"   Share actually declining: {df['is_declining'].mean():.1%}")

stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"2) Stale (180+ days) and visible (500+ impressions) pages: {len(stale_visible):,}")
print(f"   Share of those that are declining: {stale_visible['is_declining'].mean():.1%}")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

score = (df["days_since_last_update"] >= 180).astype(int) * (df["impressions_90d"] >= 500).astype(int) * df["impressions_90d"]
print(f"3) Hand rule Precision@50: {precision_at_k(score, df['is_declining'].values, 50):.2f}")
print(f"   Picking pages at random would give about: {df['is_declining'].mean():.2f}")


1) Pages in the starter data: 30,000
   Share actually declining: 54.2%
2) Stale (180+ days) and visible (500+ impressions) pages: 17
   Share of those that are declining: 94.1%
3) Hand rule Precision@50: 0.68
   Picking pages at random would give about: 0.54


## 4. Careful words: what I can and can't claim

What my work can say:

- Observed and measured statements about this dataset. For example: in the starter data, stale and visible pages were declining at 94.1%.
- Directional, decision-support claims: this ranked order is a better order to review pages in than random choice or gut feeling, checked with Precision@K and, later in the track, on clients the model never saw.
- Claims scoped to the data and the time window I actually measured, and no wider.

What my work can never say:

- That any signal causes a ranking or a traffic change. This data is observational and there is no experiment, so I will never write that I proved a Google ranking factor, or that I can predict Google's algorithm.
- That refreshing a flagged page will bring its traffic back. I can observe decline; recovery is never guaranteed.
- Anything that identifies a real client. The data is anonymized, and no client names, URLs, or private queries will appear in anything I publish.
- Any score built on leaked signals. trend_pct and trend_direction describe the outcome itself, so they are never features, only the thing being predicted.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.